# Lesson 21 Lab — Quantizing Vision and Multimodal Models

**Puzzle:** Why can a text-only calibration set miss important failure modes in a vision-language model?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

A vision-language system contains a vision encoder, patch/token embedding, projector, cross- or self-attention, language model, and KV cache. Each component sees a different activation distribution.

### Core mechanism

Patch projection maps local pixel statistics into tokens; contrast and modality shifts can create channel ranges absent from text calibration. Quantization error can then propagate through normalization and attention.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "21-multimodal-quantization"
device = require_cuda()
torch.manual_seed(2026 + 21)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Quantizing the large language component may save most bytes, while quantizing a sensitive bridge can cause disproportionate quality loss. Component-specific fallback may be cheaper than one global dtype.

### What this code tests

The notebook isolates a patch projection and compares normal versus high-contrast image distributions, carefully avoiding a full-VLM claim.

**Experiment:** Quantize a CUDA patch-projection weight and compare reconstruction error for ordinary and high-contrast synthetic images.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
conv=torch.nn.Conv2d(3,64,kernel_size=16,stride=16,bias=False,device=device); w=conv.weight.detach(); _,_,dq=symmetric_quantize(w.reshape(64,-1),bits=4,group_size=192); dq=dq.reshape_as(w)
normal=torch.randn(8,3,224,224,device=device); contrast=normal.clone(); contrast[:,:,::16,::16]*=20
def project(x,weight): return torch.nn.functional.conv2d(x,weight,stride=16)
rows={}
for name,x in {"normal":normal,"high_contrast":contrast}.items(): rows[name]=error_metrics(project(x,w),project(x,dq))
result=base_result(21,"pytorch-gpu"); result.update({"patch_projection":{"weight_shape":list(w.shape),"group_size":192},"domain_errors":rows,
    "conclusion":"Patch-projection error changed with image distribution; no full VLM quality conclusion was made."})


## 3. Inspect the evidence

Compare domain-specific errors and keep the experiment scoped to the patch projection, not an entire VLM quality claim.

### Acceptance and rollback gate

Stratify calibration/evaluation by modality, resolution, prompt length, OCR/chart cases, and component; measure component error plus end-task multimodal quality.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Patch-projection error changed with image distribution; no full VLM quality conclusion was made.",
  "domain_errors": {
    "high_contrast": {
      "cosine": 0.99766618,
      "mae": 0.04951562,
      "max_abs": 0.35846561,
      "rmse": 0.06301098
    },
    "normal": {
      "cosine": 0.99753112,
      "mae": 0.03256194,
      "max_abs": 0.20235768,
      "rmse": 0.04085071
    }
  },
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:04+00:00",
  "lesson": 21,
  "patch_projection": {
    "group_size": 192,
    "weight_shape": [
      64,
      3,
      16,
      16
    ]
  },
  "schema_version": 1
}
Saved: artifacts/rtx5090-result.json


## 4. Explain the result

Calibrate and regress each modality and bridge component rather than applying a text-only decision globally.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).